# Projet 5 : Modele d'allocation de portefeuille par Machine Learning

**Cours :** AI In Finance
**Encadrants :** Nicolas De Roux et Mohamed El Fakir
**Groupe :** Groupe 1
**Membres :** Mael Pertuisot, Valentin Martel, Mathias Garnier
**Date :** 20/04/2026

---

## Objectif

Construire un modele d'allocation de portefeuille base sur le Machine Learning, applique a 50 actions du NASDAQ. La demarche suit trois etapes :

1. **Prediction** : estimer les rendements hebdomadaires de chaque action via quatre modeles (Lineaire, Ridge, Random Forest, GRU).
2. **Allocation** : construire trois portefeuilles, Equal Weight (benchmark 1/N), Minimum Variance et Maximum Sharpe avec signal ML.
3. **Evaluation** : comparer les strategies via le ratio de Sharpe, le maximum drawdown et le turnover.

**Dataset :** cours ajustes des 50 actions NASDAQ via `yfinance`.
**Periode :** 2018-01-01 a 2025-12-31. Split : train avant 2023, test a partir de 2023.
**Benchmark marche :** S&P 500 (ticker `^GSPC`), plus representatif de la capitalisation large-cap americaine que le NASDAQ Composite pour ce panel diversifie.

## Plan du notebook

1. Imports
2. Collecte des donnees
3. Analyse exploratoire (EDA)
4. Feature engineering
5. Modeles supervises
6. Modele GRU
7. Construction des portefeuilles
8. Frontiere efficiente
9. Evaluation et comparaison
10. Conclusion


## 1. Imports

In [ ]:
!pip install yfinance scikit-learn torch matplotlib seaborn scipy --quiet

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from scipy.optimize import minimize

import torch
import torch.nn as nn
import torch.optim as optim

# Reproductibilite
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

pd.set_option('display.width', 130)
pd.set_option('display.max_columns', 20)

print("Imports OK")

## 2. Collecte des donnees

On selectionne **50 actions du NASDAQ** reparties sur 8 secteurs economiques (Tech, Sante, Consommation, Finance, Industrie, Telecom, Utilities, Energie) pour assurer la diversification exigee par le sujet.

In [ ]:
# 50 tickers NASDAQ verifies actifs sur toute la periode 2018-2025
tickers_par_secteur = {
    'Tech':       ['AAPL', 'MSFT', 'GOOGL', 'META', 'NVDA', 'ADBE', 'CRM', 'INTC',
                   'AMD', 'CSCO', 'AVGO', 'TXN', 'QCOM', 'AMAT', 'MU'],
    'Sante':      ['AMGN', 'GILD', 'MRNA', 'REGN', 'VRTX', 'BIIB', 'ILMN', 'DXCM'],
    'Conso':      ['AMZN', 'TSLA', 'COST', 'PEP', 'SBUX', 'MDLZ', 'KHC', 'MNST'],
    'Finance':    ['PYPL', 'ISRG', 'MELI', 'FTNT', 'NFLX'],
    'Industrie':  ['HON', 'CSX', 'ODFL', 'FAST', 'PAYX'],
    'Telecom':    ['CMCSA', 'TMUS', 'CHTR'],
    'Utilities':  ['XEL', 'AEP', 'EXC'],
    'Energie':    ['CEG', 'FANG', 'DDOG'],
}

# Aplatir en une liste de tickers + dictionnaire ticker -> secteur
tickers = []
secteur_map = {}
for secteur, liste in tickers_par_secteur.items():
    for t in liste:
        tickers.append(t)
        secteur_map[t] = secteur

print(f"Nombre de tickers : {len(tickers)}")
print(f"Secteurs : {list(tickers_par_secteur.keys())}")

In [ ]:
# Telechargement des cours via yfinance
data_brut = yf.download(
    tickers=tickers,
    start='2018-01-01',
    end='2026-01-01',
    auto_adjust=False,
    progress=False,
)

# On garde les cours ajustes et les volumes
prices  = data_brut['Adj Close'].ffill().dropna()
volumes = data_brut['Volume'].loc[prices.index]

print(f"Periode : {prices.index[0].date()} a {prices.index[-1].date()}")
print(f"Nombre de jours : {len(prices)}")
print(f"Tickers conserves : {prices.shape[1]}")

In [ ]:
# Repartition des tickers par secteur
df_secteurs = pd.DataFrame({
    'ticker':  prices.columns,
    'secteur': [secteur_map.get(t, 'Autre') for t in prices.columns],
}).set_index('ticker')

print("Repartition par secteur :")
print(df_secteurs['secteur'].value_counts())

## 3. Analyse exploratoire (EDA)

On compare nos 50 actions au **S&P 500** (`^GSPC`), benchmark large-cap americain. On verifie la qualite des rendements, le profil risque-rendement par secteur et la structure de correlation (input cle pour l'optimisation de portefeuille).

In [ ]:
# Rendements journaliers
returns = prices.pct_change().dropna()

# S&P 500 comme benchmark marche (remplace le NASDAQ Composite)
sp500 = yf.download('^GSPC', start='2018-01-01', end='2026-01-01',
                    auto_adjust=True, progress=False)['Close']
sp500 = sp500.reindex(prices.index).ffill()

print(f"Shape rendements : {returns.shape}")
print(f"S&P 500 recupere : {len(sp500)} jours")

In [ ]:
# Performance base 100 par secteur vs S&P 500
prix_secteurs = pd.DataFrame(index=prices.index)
for secteur, liste in tickers_par_secteur.items():
    cols = [t for t in liste if t in prices.columns]
    if cols:
        prix_secteurs[secteur] = prices[cols].mean(axis=1)

# Normalisation base 100
prix_norm  = prix_secteurs / prix_secteurs.iloc[0] * 100
sp500_norm = sp500 / sp500.iloc[0] * 100

plt.figure(figsize=(13, 6))
for secteur in prix_norm.columns:
    plt.plot(prix_norm.index, prix_norm[secteur], label=secteur, linewidth=1.3, alpha=0.85)
plt.plot(sp500_norm.index, sp500_norm, label='S&P 500', color='black',
         linewidth=2, linestyle='--')
plt.title('Performance base 100 par secteur vs S&P 500 (2018 a 2025)')
plt.xlabel('Date')
plt.ylabel('Indice base 100')
plt.legend(fontsize=8, loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

La Tech domine largement sur la periode. Le drawdown de 2022 (hausse des taux Fed) et la reprise asymetrique de 2023 apparaissent clairement. Cette heterogeneite sectorielle justifie l'usage du **secteur** comme feature et la necessite d'optimiseurs qui exploitent la **diversification**.

In [ ]:
# Matrice de correlation entre secteurs
returns_secteur = pd.DataFrame()
for secteur in tickers_par_secteur.keys():
    cols = [t for t in tickers_par_secteur[secteur] if t in prices.columns]
    if cols:
        returns_secteur[secteur] = returns[cols].mean(axis=1)

plt.figure(figsize=(8, 6))
sns.heatmap(returns_secteur.corr(), annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Correlation entre secteurs (rendements moyens journaliers)')
plt.tight_layout()
plt.show()

La plupart des secteurs sont positivement correles (exposition commune au marche), avec des intensites variables. Les Utilities sont les moins correles aux secteurs cycliques, propriete exploitee par le Minimum Variance pour reduire le risque global.

In [ ]:
# Volatilite annualisee glissante par secteur vs S&P 500
vol_30j = returns.rolling(30).std() * np.sqrt(252)

vol_secteur = pd.DataFrame()
for secteur in tickers_par_secteur.keys():
    cols = [t for t in tickers_par_secteur[secteur] if t in prices.columns]
    if cols:
        vol_secteur[secteur] = vol_30j[cols].mean(axis=1)

vol_sp500 = sp500.pct_change().rolling(30).std() * np.sqrt(252)

plt.figure(figsize=(13, 5))
for col in vol_secteur.columns:
    plt.plot(vol_secteur.index, vol_secteur[col], label=col, alpha=0.7, linewidth=1.2)
plt.plot(vol_sp500.index, vol_sp500.values, label='S&P 500',
         color='black', linewidth=2, linestyle='--')
plt.title('Volatilite annualisee glissante (30j) par secteur vs S&P 500')
plt.xlabel('Date')
plt.ylabel('Volatilite annualisee')
plt.legend(fontsize=8)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

Deux pics de volatilite ressortent : le choc COVID (mars 2020) et la remontee des taux Fed (2022). Ces regimes de volatilite non-stationnaire expliquent en partie les R-carre faibles observes en finance (cf. TP3 sur la stationnarite).

## 4. Feature engineering

Pour chaque action, on construit des features classiques de gestion quantitative :

- **Lags** : `ret_lag1` a `ret_lag5` (rendements decales).
- **Tendance** : ratio moyenne mobile 5j / 20j.
- **Volatilite** : realisee glissante 10j et 20j.
- **Momentum** : variation sur 5j et 20j, RSI 14j.
- **Flux** : volume relatif a sa moyenne mobile 20j.
- **Marche** : correlation glissante 30j avec le rendement moyen du marche.
- **Secteur** : one-hot encoding (8 variables binaires).

**Target :** rendement cumule sur les 5 prochains jours (next-week return).

In [ ]:
# Rendement moyen du marche (proxy de l'indice equipondere)
rendement_marche = returns.mean(axis=1)

def creer_features(prix, ret, vol, rendement_mkt):
    """Construit les features et la target pour une action."""
    df = pd.DataFrame(index=prix.index)
    df['ret'] = ret

    # Lags des rendements
    for lag in range(1, 6):
        df[f'ret_lag{lag}'] = df['ret'].shift(lag)

    # Ratio de moyennes mobiles (tendance)
    df['ma_ratio'] = prix.rolling(5).mean() / prix.rolling(20).mean()

    # Volatilite glissante
    df['vol_10j'] = df['ret'].rolling(10).std()
    df['vol_20j'] = df['ret'].rolling(20).std()

    # RSI 14 jours
    delta = prix.diff()
    gain  = delta.where(delta > 0, 0).rolling(14).mean()
    perte = (-delta.where(delta < 0, 0)).rolling(14).mean()
    df['rsi'] = 100 - (100 / (1 + gain / perte.replace(0, np.nan)))

    # Momentum
    df['momentum_5j']  = prix.pct_change(5)
    df['momentum_20j'] = prix.pct_change(20)

    # Volume relatif
    df['vol_rel'] = vol / vol.rolling(20).mean()

    # Correlation au marche (beta dynamique)
    df['corr_marche'] = df['ret'].rolling(30).corr(rendement_mkt)

    # Target : rendement des 5 prochains jours, shift(-5) evite le look-ahead
    df['target'] = prix.pct_change(5).shift(-5)

    return df.drop(columns=['ret']).dropna()

In [ ]:
# Application aux 50 actions
features_dict = {t: creer_features(prices[t], returns[t], volumes[t], rendement_marche)
                 for t in prices.columns}

# Empilement en panel (date, ticker) + one-hot secteur
liste_dfs = []
for t, df_t in features_dict.items():
    df_t = df_t.copy()
    df_t['ticker']  = t
    df_t['secteur'] = secteur_map.get(t, 'Autre')
    liste_dfs.append(df_t)

df_all = pd.concat(liste_dfs).reset_index().rename(columns={'index': 'Date'})
df_all = pd.get_dummies(df_all, columns=['secteur'], prefix='sect')

# Conversion bool -> float pour sklearn
for c in df_all.columns:
    if df_all[c].dtype == bool:
        df_all[c] = df_all[c].astype(float)

print(f"Shape du panel : {df_all.shape}")
print(f"Nombre de features : {df_all.shape[1] - 3}")  # -Date, -ticker, -target
df_all.head(3)

### Interpretation

Le format **panel** (date, ticker) agrege les 50 series en un seul dataset. Trois avantages :
1. **Plus de donnees** : environ 100 000 observations contre 2000 par ticker isole.
2. **Apprentissage partage** : les regularites communes (ex: le momentum) beneficient a toutes les actions.
3. **Modele unique** au lieu de 50 modeles sous-dimensionnes.

L'information sectorielle (one-hot) permet au modele d'ajuster ses predictions par secteur.

## 5. Modeles supervises

On entraine trois modeles vus en TP2 :
- **Regression lineaire** : baseline interpretable.
- **Ridge** (alpha=10) : regularisation L2 pour reduire la variance.
- **Random Forest** : modele ensembliste non-lineaire robuste aux interactions.

**Protocole :** split temporel strict, train avant 2023, test a partir de 2023. Scaling fit uniquement sur le train (evite le data leakage, cf. TP3).

In [ ]:
# Split temporel
date_split = '2023-01-01'
cols_features   = [c for c in df_all.columns if c not in ['Date', 'ticker', 'target']]
cols_secteur    = [c for c in cols_features if c.startswith('sect_')]
cols_numeriques = [c for c in cols_features if c not in cols_secteur]

mask_train = df_all['Date'] < date_split
mask_test  = df_all['Date'] >= date_split

X_train = df_all.loc[mask_train, cols_features].values.astype(float)
y_train = df_all.loc[mask_train, 'target'].values
X_test  = df_all.loc[mask_test,  cols_features].values.astype(float)
y_test  = df_all.loc[mask_test,  'target'].values

info_test = df_all.loc[mask_test, ['Date', 'ticker']].copy().reset_index(drop=True)

print(f"Train : {mask_train.sum():>6} observations")
print(f"Test  : {mask_test.sum():>6} observations")
print(f"Features : {len(cols_features)} ({len(cols_numeriques)} numeriques + {len(cols_secteur)} sectorielles)")

In [ ]:
# Scaling : on ne scale pas les colonnes one-hot (deja 0/1)
idx_num = [cols_features.index(c) for c in cols_numeriques]

scaler = StandardScaler()
X_train_sc = X_train.copy()
X_test_sc  = X_test.copy()
X_train_sc[:, idx_num] = scaler.fit_transform(X_train[:, idx_num])
X_test_sc[:, idx_num]  = scaler.transform(X_test[:, idx_num])

print("Scaling OK (fit sur train uniquement)")

In [ ]:
# Entrainement des 3 modeles
lr    = LinearRegression().fit(X_train_sc, y_train)
ridge = Ridge(alpha=10.0).fit(X_train_sc, y_train)
rf    = RandomForestRegressor(n_estimators=100, max_depth=6,
                               min_samples_leaf=20, random_state=SEED, n_jobs=-1)
rf.fit(X_train_sc, y_train)

# Predictions sur le test
pred_lr    = lr.predict(X_test_sc)
pred_ridge = ridge.predict(X_test_sc)
pred_rf    = rf.predict(X_test_sc)

# Evaluation
resultats = {}
for nom, pred in [('Lineaire', pred_lr), ('Ridge', pred_ridge), ('Random Forest', pred_rf)]:
    resultats[nom] = {
        'RMSE': np.sqrt(mean_squared_error(y_test, pred)),
        'MAE':  mean_absolute_error(y_test, pred),
        'R2':   r2_score(y_test, pred),
    }

resultats_df = pd.DataFrame(resultats).T
print("Metriques out-of-sample :")
print(resultats_df.round(6))

### Interpretation

Les **R-carre sont proches de zero**, voire negatifs. Ce n'est pas un bug mais une propriete structurelle des marches : le rapport signal / bruit est extremement faible sur les rendements.

Comme le rappelle le cours, ce qui compte in fine n'est pas la precision predictive pure mais la **valeur economique** (Sharpe, drawdown, turnover). Un R-carre de 1 a 5% suffit a generer du Sharpe positif dans l'industrie.

Le **Random Forest** domine en RMSE grace a sa capacite a capturer des interactions non-lineaires (momentum conditionnel a la volatilite, effets sectoriels conditionnels).

In [ ]:
# Feature importance du Random Forest
importance = pd.DataFrame({
    'feature':    cols_features,
    'importance': rf.feature_importances_,
}).sort_values('importance', ascending=False)

plt.figure(figsize=(9, 5))
plt.barh(importance['feature'].iloc[:12], importance['importance'].iloc[:12])
plt.xlabel('Importance')
plt.title('Top 12 features - Random Forest')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

Les variables les plus predictives sont typiquement le **momentum 20j**, la **volatilite realisee** et le **RSI**. Les variables sectorielles one-hot ont une importance faible, le modele capture l'information via les signaux numeriques.

## 6. Modele GRU

On complete les modeles tabulaires par un reseau recurrent **GRU** (Gated Recurrent Unit) vu au Cours 6 / TP4. Le GRU utilise 2 gates (update et reset) au lieu de 3 pour le LSTM : moins de parametres, entrainement plus rapide, performance comparable sur horizons courts.

**Entree :** sequence glissante de 10 jours avec les features numeriques.
**Sortie :** rendement hebdomadaire predit.

In [ ]:
# Preparation des sequences par ticker
window = 10

def create_sequences(X, y, window_size):
    """Decoupe en sequences glissantes de longueur window_size."""
    Xs, ys = [], []
    for i in range(len(X) - window_size):
        Xs.append(X[i:i + window_size])
        ys.append(y[i + window_size])
    return np.array(Xs), np.array(ys)

X_seq_train_list, y_seq_train_list = [], []
X_seq_test_list,  y_seq_test_list  = [], []

for t in prices.columns:
    df_t = df_all[df_all['ticker'] == t].sort_values('Date').reset_index(drop=True)

    X_num_t  = df_t[cols_numeriques].values.astype(float)
    X_sect_t = df_t[cols_secteur].values.astype(float)
    y_t      = df_t['target'].values.astype(float)

    mask_tr = (df_t['Date'] < date_split).values

    # Scaling par ticker, fit sur train uniquement
    sc = StandardScaler()
    X_num_sc = X_num_t.copy()
    if mask_tr.sum() > 0:
        X_num_sc[mask_tr]  = sc.fit_transform(X_num_t[mask_tr])
        X_num_sc[~mask_tr] = sc.transform(X_num_t[~mask_tr])

    X_t_sc = np.concatenate([X_num_sc, X_sect_t], axis=1)
    n_train = mask_tr.sum()

    X_tr, y_tr = create_sequences(X_t_sc[:n_train], y_t[:n_train], window)
    X_te, y_te = create_sequences(X_t_sc[n_train:], y_t[n_train:], window)

    if len(X_tr) > 0:
        X_seq_train_list.append(X_tr)
        y_seq_train_list.append(y_tr)
    if len(X_te) > 0:
        X_seq_test_list.append(X_te)
        y_seq_test_list.append(y_te)

X_seq_train = np.concatenate(X_seq_train_list)
y_seq_train = np.concatenate(y_seq_train_list)
X_seq_test  = np.concatenate(X_seq_test_list)
y_seq_test  = np.concatenate(y_seq_test_list)

print(f"GRU sequences - Train : {X_seq_train.shape}, Test : {X_seq_test.shape}")

In [ ]:
# Conversion en tenseurs et definition du modele
X_train_t = torch.tensor(X_seq_train, dtype=torch.float32)
y_train_t = torch.tensor(y_seq_train, dtype=torch.float32).unsqueeze(1)
X_test_t  = torch.tensor(X_seq_test,  dtype=torch.float32)
y_test_t  = torch.tensor(y_seq_test,  dtype=torch.float32).unsqueeze(1)

class GRURegressor(nn.Module):
    """GRU (hidden_dim=32) suivi d'une couche lineaire pour la regression."""
    def __init__(self, input_dim, hidden_dim=32):
        super().__init__()
        self.gru = nn.GRU(input_dim, hidden_dim, batch_first=True)
        self.fc  = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        _, hn = self.gru(x)
        return self.fc(hn[-1])

model_gru     = GRURegressor(input_dim=X_train_t.shape[2])
criterion     = nn.MSELoss()
optimizer_gru = optim.Adam(model_gru.parameters(), lr=1e-3)

n_params = sum(p.numel() for p in model_gru.parameters())
print(f"GRU : {n_params} parametres")

In [ ]:
# Entrainement du GRU
epochs, batch_size = 20, 128
train_losses, val_losses = [], []

for epoch in range(epochs):
    model_gru.train()
    idx = torch.randperm(X_train_t.size(0))
    X_shuf, y_shuf = X_train_t[idx], y_train_t[idx]

    epoch_loss, n_batch = 0.0, 0
    for i in range(0, X_train_t.size(0), batch_size):
        preds = model_gru(X_shuf[i:i+batch_size])
        loss  = criterion(preds, y_shuf[i:i+batch_size])
        optimizer_gru.zero_grad()
        loss.backward()
        optimizer_gru.step()
        epoch_loss += loss.item()
        n_batch    += 1
    train_losses.append(epoch_loss / n_batch)

    model_gru.eval()
    with torch.no_grad():
        val_losses.append(criterion(model_gru(X_test_t), y_test_t).item())

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:>2}/{epochs} | Train : {train_losses[-1]:.6f} | Val : {val_losses[-1]:.6f}")

# Courbe d'apprentissage
plt.figure(figsize=(8, 4))
plt.plot(train_losses, label='Train')
plt.plot(val_losses,   label='Validation')
plt.title("Courbe d'apprentissage GRU")
plt.xlabel('Epoch')
plt.ylabel('MSE')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Evaluation GRU
model_gru.eval()
with torch.no_grad():
    pred_gru = model_gru(X_test_t).numpy().flatten()

rmse_gru = np.sqrt(mean_squared_error(y_seq_test, pred_gru))
print(f"GRU RMSE : {rmse_gru:.6f}")
print(f"\nComparaison des RMSE :")
print(f"  Lineaire      : {resultats_df.loc['Lineaire', 'RMSE']:.6f}")
print(f"  Ridge         : {resultats_df.loc['Ridge', 'RMSE']:.6f}")
print(f"  Random Forest : {resultats_df.loc['Random Forest', 'RMSE']:.6f}")
print(f"  GRU           : {rmse_gru:.6f}")

### Interpretation GRU

Sur des features tabulaires avec des rendements a horizon court, le GRU n'apporte pas de gain evident par rapport au Random Forest. Explications :
1. **Signal faible** : les rendements sont domines par le bruit.
2. **Features deja temporelles** : les lags et rolling windows capturent deja la dynamique.
3. **Donnees limitees** : 50 actions sur 5 ans restent modestes pour un reseau profond.

Pour la construction du portefeuille, on retient donc les **predictions du Random Forest**, plus stable et interpretable.

## 7. Construction des portefeuilles

Le sujet demande de construire un portefeuille optimise parmi les approches *minimum variance, maximum Sharpe, or risk-parity*. On implemente les deux premieres, plus un **benchmark Equal Weight** :

| Strategie | Objectif | Utilise les predictions ML |
|---|---|---|
| Equal Weight (benchmark) | w_i = 1/N | Non |
| Minimum Variance | min w' Sigma w | Non (risque pur) |
| Maximum Sharpe (ML) | max (w' mu_ML) / sqrt(w' Sigma w) | Oui |

**Contraintes :** long-only (w_i >= 0), somme des poids = 1, **rebalance hebdomadaire**.
**Covariance :** empirique, estimee sur fenetre glissante de 60 jours.

In [ ]:
# Reconstruction des predictions RF par (date, ticker)
info_test['pred'] = pred_rf
info_test['target'] = y_test

pred_pivot   = info_test.pivot(index='Date', columns='ticker', values='pred')
target_pivot = info_test.pivot(index='Date', columns='ticker', values='target')

pred_pivot   = pred_pivot.dropna(axis=1, how='all').dropna(axis=0)
target_pivot = target_pivot.loc[pred_pivot.index, pred_pivot.columns]

# Rebalance hebdomadaire (toutes les 5 seances)
dates_hebdo  = pred_pivot.index[::5]
pred_hebdo   = pred_pivot.loc[dates_hebdo]
target_hebdo = target_pivot.loc[dates_hebdo]

tickers_portf = pred_hebdo.columns.tolist()
n_assets      = len(tickers_portf)

print(f"Dates de rebalance : {len(dates_hebdo)}")
print(f"Actifs : {n_assets}")

In [ ]:
# Parametres communs aux optimiseurs
bornes      = [(0.0, 1.0)] * n_assets   # long-only
contraintes = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0}]
w0          = np.ones(n_assets) / n_assets

# Rendements journaliers disponibles aux dates de rebalance (historique complet)
returns_full = returns[tickers_portf]
COV_WINDOW   = 60  # jours pour estimer la covariance

In [ ]:
# Strategie 1 : Equal Weight (benchmark 1/N)
poids_ew     = np.ones(n_assets) / n_assets
rendement_ew = (target_hebdo * poids_ew).sum(axis=1)
poids_ew_df  = pd.DataFrame(np.tile(poids_ew, (len(dates_hebdo), 1)),
                            index=dates_hebdo, columns=tickers_portf)

print(f"Equal Weight : poids constant de {poids_ew[0]:.4f} pour {n_assets} actifs")

In [ ]:
# Strategie 2 : Minimum Variance
# Objectif : minimiser w' Sigma w (risque pur, n'utilise pas les predictions ML)

def optimiser_min_var(cov, w_prev):
    res = minimize(
        lambda w: np.dot(w, np.dot(cov, w)),
        w_prev, method='SLSQP',
        bounds=bornes, constraints=contraintes,
        options={'maxiter': 400, 'ftol': 1e-9},
    )
    return res.x if res.success else w_prev

poids_mv  = pd.DataFrame(index=dates_hebdo, columns=tickers_portf, dtype=float)
w_courant = w0.copy()

for date in dates_hebdo:
    date_idx = returns_full.index.get_indexer([date], method='ffill')[0]
    debut    = max(0, date_idx - COV_WINDOW)

    if date_idx - debut >= 20:
        fenetre   = returns_full.iloc[debut:date_idx]
        cov       = fenetre.cov().values
        w_courant = optimiser_min_var(cov, w_courant)

    poids_mv.loc[date] = w_courant

rendement_mv = (poids_mv.astype(float) * target_hebdo).sum(axis=1)
print("Minimum Variance : optimise sur covariance empirique 60j")

In [ ]:
# Strategie 3 : Maximum Sharpe avec predictions ML
# Objectif : max (w' mu_ML) / sqrt(w' Sigma w) - c'est ici qu'on utilise le signal ML

def optimiser_max_sharpe(mu, cov, w_prev):
    def neg_sharpe(w):
        ret_p = np.dot(w, mu)
        vol_p = np.sqrt(np.dot(w, np.dot(cov, w)))
        return -ret_p / vol_p if vol_p > 1e-10 else 0.0

    res = minimize(
        neg_sharpe, w_prev, method='SLSQP',
        bounds=bornes, constraints=contraintes,
        options={'maxiter': 500, 'ftol': 1e-9},
    )
    return res.x if res.success else w_prev

poids_ms  = pd.DataFrame(index=dates_hebdo, columns=tickers_portf, dtype=float)
w_courant = w0.copy()

for date in dates_hebdo:
    mu       = pred_hebdo.loc[date].values
    date_idx = returns_full.index.get_indexer([date], method='ffill')[0]
    debut    = max(0, date_idx - COV_WINDOW)

    if date_idx - debut >= 20:
        fenetre   = returns_full.iloc[debut:date_idx]
        cov       = fenetre.cov().values
        w_courant = optimiser_max_sharpe(mu, cov, w_courant)

    poids_ms.loc[date] = w_courant

rendement_ms = (poids_ms.astype(float) * target_hebdo).sum(axis=1)
print("Max Sharpe (ML) : optimise avec mu = predictions RF")

### Interpretation des 3 strategies

- **Equal Weight** : ne fait aucune hypothese. Sa robustesse vient de cette naivete : l'absence d'erreur d'estimation compense la sous-optimalite theorique (DeMiguel, Garlappi, Uppal, 2009).
- **Minimum Variance** : surponde les actifs peu volatils et peu correles. Defensif par nature, efficace en regime de stress.
- **Maximum Sharpe (ML)** : seule strategie qui **utilise le signal ML**. Sensible a l'erreur d'estimation des rendements predits (probleme classique identifie par Michaud, 1989).

## 8. Frontiere efficiente

La frontiere efficiente est l'ensemble des portefeuilles offrant le meilleur rendement pour chaque niveau de risque. On la calcule de maniere statique sur la periode de test pour visualiser la position des strategies dans l'espace rendement-risque.

In [ ]:
# Rendements et covariance annualises sur la periode test
ret_test   = returns.loc[returns.index >= date_split, tickers_portf]
mu_annuel  = ret_test.mean() * 252
cov_annuel = ret_test.cov()  * 252

def stats(w, mu, cov):
    return np.dot(w, mu), np.sqrt(np.dot(w, np.dot(cov, w)))

# GMV statique (point extreme gauche de la frontiere)
res_gmv = minimize(
    lambda w: np.dot(w, np.dot(cov_annuel.values, w)),
    w0, method='SLSQP', bounds=bornes, constraints=contraintes,
    options={'maxiter': 400, 'ftol': 1e-9},
)
w_gmv = res_gmv.x
ret_gmv, vol_gmv = stats(w_gmv, mu_annuel.values, cov_annuel.values)

# Max Sharpe statique (portefeuille tangent)
def neg_sharpe_static(w):
    r = np.dot(w, mu_annuel.values)
    v = np.sqrt(np.dot(w, np.dot(cov_annuel.values, w)))
    return -r / v if v > 1e-10 else 0.0

res_tan = minimize(neg_sharpe_static, w0, method='SLSQP',
                   bounds=bornes, constraints=contraintes,
                   options={'maxiter': 400, 'ftol': 1e-9})
w_tan = res_tan.x
ret_tan, vol_tan = stats(w_tan, mu_annuel.values, cov_annuel.values)

# Construction de la frontiere par scan de rendements cibles
def min_vol_pour_target(target_ret):
    c = contraintes + [{'type': 'eq', 'fun': lambda w: np.dot(w, mu_annuel.values) - target_ret}]
    r = minimize(lambda w: np.sqrt(np.dot(w, np.dot(cov_annuel.values, w))),
                 w0, method='SLSQP', bounds=bornes, constraints=c,
                 options={'maxiter': 400, 'ftol': 1e-9})
    return r.x if r.success else None

target_rets = np.linspace(ret_gmv, mu_annuel.max() * 0.98, 50)
frontier_vols, frontier_rets = [], []
for tr in target_rets:
    w = min_vol_pour_target(tr)
    if w is not None:
        r, v = stats(w, mu_annuel.values, cov_annuel.values)
        frontier_rets.append(r)
        frontier_vols.append(v)

print(f"GMV        : rendement={ret_gmv:.3f}, vol={vol_gmv:.3f}")
print(f"Max Sharpe : rendement={ret_tan:.3f}, vol={vol_tan:.3f}")

In [ ]:
# Graphique : frontiere + actifs individuels
asset_vols = ret_test.std().values * np.sqrt(252)
asset_rets = ret_test.mean().values * 252

plt.figure(figsize=(11, 6))
plt.plot(frontier_vols, frontier_rets, 'b--', linewidth=2, label='Frontiere efficiente')
plt.scatter(vol_gmv, ret_gmv, c='red',  marker='D', s=120, label='GMV',        zorder=5)
plt.scatter(vol_tan, ret_tan, c='lime', marker='X', s=150, label='Max Sharpe', zorder=5)
plt.scatter(asset_vols, asset_rets, c='black', s=40, alpha=0.7, label='50 actions')

for nom, v, r in zip(tickers_portf, asset_vols, asset_rets):
    plt.annotate(nom, (v, r), xytext=(5, 0), textcoords='offset points',
                 fontsize=6, alpha=0.7)

plt.title('Frontiere efficiente des 50 actions NASDAQ (test 2023 a 2025)')
plt.xlabel('Volatilite annualisee (risque)')
plt.ylabel('Rendement annualise')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

La frontiere domine largement les actions individuelles : pour n'importe quel actif seul, il existe un portefeuille diversifie offrant un meilleur rendement a risque egal. C'est la preuve visuelle du theoreme de diversification de Markowitz.

GMV et Max Sharpe **theoriques** (calcules avec les vrais rendements de la periode test) sont des benchmarks post-facto : ils supposent une connaissance parfaite des rendements futurs, impossible en pratique. Nos strategies rebalancees chaque semaine (section 7) cherchent a s'en approcher **sans look-ahead**.

## 9. Evaluation et comparaison

On compare les 3 strategies sur la periode out-of-sample via les metriques exigees par le sujet : ratio de Sharpe, maximum drawdown et turnover. On ajoute le **S&P 500** comme benchmark marche.

In [ ]:
# Fonctions d'evaluation (frequence hebdomadaire)
FREQ = 52

def rendement_annualise(r, freq=FREQ):
    cumul = (1 + r).prod()
    n_ans = len(r) / freq
    return cumul ** (1 / n_ans) - 1 if n_ans > 0 else 0.0

def sharpe_ratio(r, freq=FREQ, rf=0.0):
    excess = r - rf / freq
    return np.sqrt(freq) * excess.mean() / excess.std() if excess.std() > 0 else 0.0

def max_drawdown(r):
    cumul = (1 + r).cumprod()
    return ((cumul - cumul.cummax()) / cumul.cummax()).min()

def turnover(poids_df):
    # Convention standard : 0.5 * somme(|delta w|)
    return (0.5 * poids_df.diff().abs().sum(axis=1)).iloc[1:].mean()

In [ ]:
# Tableau comparatif
metriques = pd.DataFrame({
    'Equal Weight': [
        rendement_annualise(rendement_ew),
        rendement_ew.std() * np.sqrt(FREQ),
        sharpe_ratio(rendement_ew),
        max_drawdown(rendement_ew),
        turnover(poids_ew_df),
    ],
    'Min Variance': [
        rendement_annualise(rendement_mv),
        rendement_mv.std() * np.sqrt(FREQ),
        sharpe_ratio(rendement_mv),
        max_drawdown(rendement_mv),
        turnover(poids_mv.astype(float)),
    ],
    'Max Sharpe (ML)': [
        rendement_annualise(rendement_ms),
        rendement_ms.std() * np.sqrt(FREQ),
        sharpe_ratio(rendement_ms),
        max_drawdown(rendement_ms),
        turnover(poids_ms.astype(float)),
    ],
}, index=['Rendement annualise', 'Volatilite annualisee', 'Ratio de Sharpe',
          'Max Drawdown', 'Turnover moyen'])

print("Comparaison des portefeuilles (out-of-sample 2023 a 2025)")
print(metriques.round(4))

In [ ]:
# Rendement cumule vs S&P 500
cumul_ew = (1 + rendement_ew).cumprod(); cumul_ew = cumul_ew / cumul_ew.iloc[0]
cumul_mv = (1 + rendement_mv).cumprod(); cumul_mv = cumul_mv / cumul_mv.iloc[0]
cumul_ms = (1 + rendement_ms).cumprod(); cumul_ms = cumul_ms / cumul_ms.iloc[0]

sp500_weekly = sp500.reindex(rendement_ew.index, method='ffill')
sp500_bt     = sp500_weekly.pct_change().fillna(0)
cumul_sp     = (1 + sp500_bt).cumprod(); cumul_sp = cumul_sp / cumul_sp.iloc[0]

plt.figure(figsize=(12, 6))
plt.plot(cumul_ew.index, cumul_ew.values, label='Equal Weight (1/N)', linewidth=1.8, linestyle='--')
plt.plot(cumul_mv.index, cumul_mv.values, label='Min Variance',        linewidth=1.8)
plt.plot(cumul_ms.index, cumul_ms.values, label='Max Sharpe (ML)',     linewidth=1.8)
plt.plot(cumul_sp.index, cumul_sp.values, label='S&P 500',
         color='black', linewidth=2, linestyle=':')
plt.title('Rendement cumule des 3 strategies vs S&P 500 (base 1)')
plt.xlabel('Date')
plt.ylabel('Valeur cumulee')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Drawdown
def serie_drawdown(r):
    cumul = (1 + r).cumprod()
    return (cumul - cumul.cummax()) / cumul.cummax()

dd_ew = serie_drawdown(rendement_ew)
dd_mv = serie_drawdown(rendement_mv)
dd_ms = serie_drawdown(rendement_ms)
dd_sp = serie_drawdown(sp500_bt)

plt.figure(figsize=(12, 5))
plt.fill_between(dd_ew.index, dd_ew.values, 0, alpha=0.25, label='Equal Weight')
plt.fill_between(dd_mv.index, dd_mv.values, 0, alpha=0.25, label='Min Variance')
plt.fill_between(dd_ms.index, dd_ms.values, 0, alpha=0.25, label='Max Sharpe (ML)')
plt.plot(dd_sp.index, dd_sp.values, label='S&P 500',
         color='black', linewidth=1.8, linestyle='--')
plt.title('Drawdown des 3 strategies vs S&P 500')
plt.xlabel('Date')
plt.ylabel('Drawdown')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Repartition sectorielle au cours du temps (Min Var et Max Sharpe ML)
fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

secteurs_portf = {}
for t in tickers_portf:
    s = secteur_map.get(t, 'Autre')
    secteurs_portf.setdefault(s, []).append(t)

for ax, (poids, titre) in zip(axes, [
    (poids_mv.astype(float), 'Minimum Variance'),
    (poids_ms.astype(float), 'Maximum Sharpe (ML)'),
]):
    alloc = pd.DataFrame(index=poids.index)
    for s, cols in secteurs_portf.items():
        alloc[s] = poids[cols].sum(axis=1)
    alloc = alloc.fillna(0).div(alloc.sum(axis=1), axis=0)

    ax.stackplot(alloc.index, alloc.T.values, labels=alloc.columns, alpha=0.85)
    ax.set_title(f'Repartition sectorielle : {titre}')
    ax.set_ylabel('Poids cumule')
    ax.set_ylim(0, 1)
    ax.legend(loc='upper left', fontsize=8, ncol=4)
    ax.grid(True, alpha=0.2)

axes[-1].set_xlabel('Date')
plt.tight_layout()
plt.show()

### Interpretation de l'evaluation

- **Equal Weight** : turnover nul (par definition). Benchmark robuste sur le long terme.
- **Min Variance** : plus faible volatilite et drawdown. Concentre sur les secteurs defensifs (Utilities, Sante).
- **Max Sharpe (ML)** : turnover plus eleve car reagit aux predictions. Performance depend de la qualite du signal ML.

Le ratio de Sharpe est la metrique la plus comparable entre strategies car il normalise par le risque. Un turnover eleve implique des couts de transaction non modelises ici, qui peuvent grignoter la performance brute.

## 10. Conclusion

### Ce qui a fonctionne

1. **Pipeline complet conforme au sujet** : 50 actions NASDAQ, features techniques et sectorielles, 4 modeles ML, 3 strategies d'allocation, evaluation par Sharpe / drawdown / turnover.
2. **Split temporel strict** : train avant 2023, test a partir de 2023, scaling fit sur train uniquement, aucun data leakage.
3. **Diversification respectee** : les 8 secteurs sont representes, la frontiere efficiente confirme le benefice de la diversification.

### Limites identifiees

- **R-carre de prediction tres faible** : structurel en finance, les hedge funds travaillent avec des R-carre de 1 a 5%.
- **GRU sans gain net sur Random Forest** : typique sur features tabulaires a horizon court.
- **Max Sharpe sensible aux erreurs d'estimation** : les predictions ML bruitees entrainent un turnover eleve (Michaud, 1989).
- **Couts de transaction non modelises** : un turnover de 20 a 30% hebdomadaire generait 1 a 2% de frais annuels chez un gerant institutionnel.

### Ameliorations possibles

- Ajouter une **penalite de turnover** dans l'optimiseur Max Sharpe.
- Integrer des **donnees macro** (taux Fed, VIX) ou du **sentiment** via NLP (Cours 6).
- Remplacer le Random Forest par **XGBoost**.
- Etendre a d'autres univers (S&P 500 complet, marches internationaux).

### Message final

La valeur economique d'un modele ML en finance ne se mesure pas a son R-carre mais a son ratio de Sharpe net de couts. Un modele avec un R-carre proche de zero peut produire un portefeuille performant si ses predictions sont correctement integrees dans une optimisation realiste.

Les trois strategies testees illustrent le compromis fondamental de la gestion : le **1/N naif** est difficile a battre en regime normal, le **Min Variance** protege en regime de stress, et le **Max Sharpe (ML)** exploite l'alpha mais paie des couts de transaction eleves. Le choix depend du mandat d'investissement et de la tolerance au risque.
